# 🏁 Day 7 Assignment Solution: Command-Line Resume Analyzer

Welcome to the reference solution for the **Day 7 Assignment**. 
This notebook contains the complete, runnable implementation of the **TalentScout Resume Analyzer**.

### 🏗️ Solution Overview
This pipeline uses:
1. **System Instruction**: Enforcing an elite, objective recruiter persona.
2. **Pydantic Model Schema**: Enforcing structured JSON responses for candidate details, match scores, and skill gaps.
3. **Function Calling**: Defining a local function `get_upskilling_resources` to fetch course/project recommendations for missing skills.
4. **Interactive CLI**: Managing interactive commands (`analyze`, `stats`, `save`, `quit`) and tracking session costs.

## 🛠️ Step 1: Install & Configure Dependencies

In [ ]:
import os
import json
from typing import Optional, List
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import google.generativeai as genai

# Load environment variables from .env file
load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found. Please verify your .env file.")

genai.configure(api_key=API_KEY)
print("✅ Gemini API successfully initialized!")

## 📦 Step 2: Define Pydantic Output Schemas

We define the contract for the structured recruiter output using Pydantic. This structure will be enforced directly at the model's generation stage.

In [ ]:
class ResumeAnalysis(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate")
    email: str = Field(description="Candidate email address extracted from resume")
    phone: str = Field(description="Candidate contact phone number extracted")
    current_role: str = Field(description="Candidate's current or last professional job title")
    skills: List[str] = Field(description="List of technical skills explicitly listed or clearly demonstrated")
    years_of_experience: float = Field(description="Total years of professional experience calculated from employment dates")
    
    match_score: int = Field(description="Alignment percentage score (0 to 100) comparing candidate skills to the Job Description")
    missing_skills: List[str] = Field(description="List of required technical skills present in the JD but missing from the resume")
    alignment_explanation: str = Field(description="Concise explanation of the matching score. Maximum 3 sentences.")
    
    recommended_resources: List[str] = Field(default=[], description="List of suggested courses and projects retrieved from database tools. Leave empty initially.")

## 🛠️ Step 3: Define Local Database Upskilling Tool

We build a Python function that maps technical topics to courses and projects. We will register this function in the LLM's tools context.

In [ ]:
def get_upskilling_resources(missing_skills: list[str]) -> str:
    """Queries the TalentScout database to retrieve training courses and 
    practical projects for a list of missing technical skills.
    
    Args:
        missing_skills: A list of skill names (e.g. ['Docker', 'FastAPI'])
    """
    # Database mapping skills to learning material
    resource_db = {
        "docker": "[Course] Docker Deep Dive (Udemy) | [Project] Containerize a Python FastAPI Microservice with multi-stage builds",
        "kubernetes": "[Course] Certified Kubernetes Administrator - CKA (KodeKloud) | [Project] Deploy a replica-set cluster on Minikube with Ingress routing",
        "fastapi": "[Course] Building APIs with FastAPI (TalkPython) | [Project] Build a secure async CRUD API using SQLModel and SQLite",
        "pytorch": "[Course] Deep Learning with PyTorch (Udacity) | [Project] Train a Custom Image Classifier with ResNet transfer learning",
        "react": "[Course] React - The Complete Guide (Academind) | [Project] Build a dynamic analytics dashboard with TailwindCSS and Vite",
        "aws": "[Course] AWS Solutions Architect Associate (A Cloud Guru) | [Project] Deploy a highly-available auto-scaling web application using EC2 and RDS",
        "github actions": "[Course] CI/CD with GitHub Actions (GitHub) | [Project] Setup automated testing, linting and docker image publishing on main push",
        "postgresql": "[Course] Complete SQL bootcamp (Udemy) | [Project] Perform database optimization using indexes and query execution plans"
    }
    
    recommendations = []
    for skill in missing_skills:
        clean_skill = skill.strip().lower()
        if clean_skill in resource_db:
            recommendations.append(f"{skill}: {resource_db[clean_skill]}")
        else:
            recommendations.append(f"{skill}: [Course] Basic {skill} foundations | [Project] Build a core starter project utilizing {skill}")
            
    return json.dumps({"recommendations": recommendations})

## 🤖 Step 4: Implement Recruiter Agent Engine

We create a class `ResumeAnalyzerAgent` to handle prompt formatting, tool invocation, token accounting, and cost tracking.

In [ ]:
class ResumeAnalyzerAgent:
    def __init__(self):
        self.system_prompt = (
            "You are an elite technical recruiter at TalentScout. "
            "You evaluate candidate resumes against target job descriptions objectively and thoroughly. "
            "First, parse the candidate profile details. Next, compare the candidate's skills to the JD, "
            "score the match out of 100, and list missing skills. "
            "If the candidate has missing skills, you MUST query the get_upskilling_resources tool "
            "with the list of missing skills to retrieve recommendations. "
            "Finally, output the structured details matching the ResumeAnalysis schema exactly."
        )
        
        # Initialize conversational model equipped with tools
        # enable_automatic_function_calling resolves the client function execution automatically
        self.model = genai.GenerativeModel(
            model_name="gemini-1.5-flash",
            system_instruction=self.system_prompt,
            tools=[get_upskilling_resources]
        )
        
        # Initialize pricing tracking variables
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        
    def analyze(self, resume_text: str, jd_text: str) -> ResumeAnalysis:
        self.total_api_calls += 1
        
        # Chat session holds tools memory loop
        chat = self.model.start_chat(enable_automatic_function_calling=True)
        
        prompt = (
            f"Target Job Description:\n{jd_text}\n\n"
            f"Candidate Resume Text:\n{resume_text}\n\n"
            "Analyze and return structured details according to the required schema contract."
        )
        
        response = chat.send_message(
            prompt,
            generation_config=genai.types.GenerationConfig(
                response_mime_type="application/json",
                response_schema=ResumeAnalysis,
                temperature=0.0
            )
        )
        
        # Record Token metrics
        if response.usage_metadata:
            self.total_input_tokens += response.usage_metadata.prompt_token_count
            self.total_output_tokens += response.usage_metadata.candidates_token_count
            
        # Load and parse output schema
        return ResumeAnalysis.model_validate_json(response.text)
        
    def get_stats(self) -> dict:
        # Pricing per 1M tokens: input=$0.075, output=$0.30
        input_cost = (self.total_input_tokens / 1_000_000) * 0.075
        output_cost = (self.total_output_tokens / 1_000_000) * 0.30
        total_cost = input_cost + output_cost
        
        return {
            "api_calls": self.total_api_calls,
            "input_tokens": self.total_input_tokens,
            "output_tokens": self.total_output_tokens,
            "estimated_cost_usd": total_cost
        }

## 💻 Step 5: Interactive Command-Line Console Interface

Now we write the shell interface that prompts recruiters for input and handles options like statistics tracking and report exports.

In [ ]:
def run_cli_app():
    agent = ResumeAnalyzerAgent()
    last_report: Optional[ResumeAnalysis] = None
    
    print("=" * 50)
    print("     TALENTSCOUT: AI RESUME ANALYZER ENGINE (CLI)     ")
    print("=" * 50)
    
    while True:
        print("\nAvailable Commands:")
        print("  analyze - Parse a resume and JD")
        print("  stats   - Show API token usage stats & costs")
        print("  save    - Save the last analysis report to JSON")
        print("  quit    - Exit application")
        
        cmd = input("\nEnter command: ").strip().lower()
        
        if cmd == "quit":
            print("👋 Goodbye!")
            break
            
        elif cmd == "stats":
            stats = agent.get_stats()
            print("\n📊 Session Analytics:")
            print(f"  Total API Calls:      {stats['api_calls']}")
            print(f"  Input Tokens Consumed: {stats['input_tokens']}")
            print(f"  Output Tokens Consumed:{stats['output_tokens']}")
            print(f"  Estimated Session Cost: ${stats['estimated_cost_usd']:.6f} USD")
            
        elif cmd == "save":
            if not last_report:
                print("⚠️ No report found. Please run 'analyze' first.")
                continue
            
            filename = f"parsed_resume_{last_report.candidate_name.replace(' ', '_')}.json"
            try:
                with open(filename, "w") as f:
                    f.write(last_report.model_dump_json(indent=2))
                print(f"💾 Report exported successfully as: {filename}")
            except Exception as e:
                print(f"❌ Failed to save report: {str(e)}")
                
        elif cmd == "analyze":
            print("\n--- RESUME ANALYSIS SETUP ---")
            print("(Paste text and press ENTER when finished)")
            
            # Prompt for resume
            print("\nEnter Resume Text:")
            resume_lines = []
            while True:
                line = input()
                if line == "":
                    break
                resume_lines.append(line)
            resume_text = "\n".join(resume_lines)
            
            if not resume_text.strip():
                print("❌ Resume text cannot be empty.")
                continue
                
            # Prompt for Job Description
            print("Enter Job Description Text:")
            jd_lines = []
            while True:
                line = input()
                if line == "":
                    break
                jd_lines.append(line)
            jd_text = "\n".join(jd_lines)
            
            if not jd_text.strip():
                print("❌ Job description cannot be empty.")
                continue
                
            print("\n🔄 Analyzing resume against Job Description... Please wait...")
            try:
                last_report = agent.analyze(resume_text, jd_text)
                print("\n================ ANALYSIS REPORT =================")
                print(last_report.model_dump_json(indent=2))
                print("==================================================")
            except Exception as e:
                print(f"💥 Analysis failed: {str(e)}")
                
        else:
            print("❌ Unknown command. Please select a valid menu command.")

## 🧪 Step 6: Test Case Execution

Let's test the recruiter pipeline directly in the notebook using a mock candidate resume (Amit Sharma, junior engineer with python/SQL background) applying for a **Senior DevOps Engineer** position requiring **Docker**, **Kubernetes**, and **GitHub Actions**.

In [ ]:
mock_resume = (
    "Amit Sharma\n"
    "Email: amit.sharma@example.com | Phone: +91-9988776655\n"
    "Role: Junior Python Developer\n"
    "Experience: 2 years\n"
    "Skills: Python, Django, PostgreSQL, Git, SQL, basic HTML\n"
    "Employment History:\n"
    "- TechCorp Solutions, Junior Dev (2024 - Present)\n"
    "  Maintained Django backends and optimization SQL queries."
)

mock_jd = (
    "Title: Senior DevOps & Cloud Infrastructure Specialist\n"
    "Core Requirements:\n"
    "- 5+ years of software/infrastructure engineering experience\n"
    "- Deep understanding of Containerization (Docker, Kubernetes)\n"
    "- Experience setting up CI/CD pipelines using GitHub Actions\n"
    "- Familiarity with cloud platforms (AWS preferred) and PostgreSQL databases."
)

agent_test = ResumeAnalyzerAgent()
report = agent_test.analyze(mock_resume, mock_jd)

print("📊 Recruiter Structured Evaluation Output:")
print(json.dumps(report.model_dump(), indent=2))

print(f"\nEstimated Cost for this single test: ${agent_test.get_stats()['estimated_cost_usd']:.6f} USD")

## 🚀 Step 7: Launch Interactive CLI App

Run the cell below to launch the interactive prompt tool directly in your Jupyter environment. You can enter commands like `analyze`, `stats`, `save`, and `quit`.

In [ ]:
# Run CLI Tool inside Jupyter
# run_cli_app()